# Paper 13 · SimCLR

**Citation:** Ting Chen et al., “A Simple Framework for Contrastive Learning of Visual Representations” (2020).

**Paper:** https://arxiv.org/abs/2002.05709

> **Scale gap:** We use small digits and simple noise/masking augmentations rather than ImageNet-scale self-supervised training.

## Mathematical Framework

Before reproducing the paper experimentally, work through:

- [Math 01 · Linear Algebra & Geometry](../../math/01_linear_algebra_geometry.ipynb)
- [Math 05 · Information Theory](../../math/05_information_theory.ipynb)
- [Math 06 · Optimization](../../math/06_optimization.ipynb)
- [Math 10 · Neural-Network Mathematics](../../math/10_neural_network_math.ipynb)

Your explanation should connect the paper's empirical claim to its **mathematical objective, representation, assumptions, and optimization/statistical argument**.

## Before you read
1. What defines a positive pair?
2. Why are in-batch negatives useful?
3. Why can augmentation choice determine what invariances the representation learns?

## Central claim
Strong augmentations plus a contrastive objective can learn useful visual representations without class labels.

In [ ]:
import numpy as np, torch, matplotlib.pyplot as plt
from torch import nn
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
d=load_digits(); X=torch.tensor((d.data/16).astype("float32")); labels=d.target
torch.manual_seed(0)
def augment(x):
    y=x+.08*torch.randn_like(x)
    mask=(torch.rand_like(y)>.08).float()
    return (y*mask).clamp(0,1)

## Encoder and NT-Xent-style objective

In [ ]:
enc=nn.Sequential(nn.Linear(64,64),nn.ReLU(),nn.Linear(64,16))
opt=torch.optim.Adam(enc.parameters(),lr=.01)
def loss_fn(z1,z2,temp=.2):
    z1=nn.functional.normalize(z1,dim=1); z2=nn.functional.normalize(z2,dim=1)
    logits=z1@z2.T/temp
    target=torch.arange(len(z1))
    return (nn.functional.cross_entropy(logits,target)+nn.functional.cross_entropy(logits.T,target))/2

for step in range(250):
    ids=torch.randint(0,len(X),(256,))
    v1,v2=augment(X[ids]),augment(X[ids])
    loss=loss_fn(enc(v1),enc(v2))
    opt.zero_grad(); loss.backward(); opt.step()
print("contrastive loss",float(loss))

## Figure-inspired embedding visualization

In [ ]:
with torch.no_grad(): Z=enc(X).numpy()
P=PCA(2).fit_transform(Z)
plt.scatter(P[:,0],P[:,1],c=labels,s=6,cmap="tab10"); plt.title("Unlabeled contrastive embeddings, colored only for inspection"); plt.show()

### Ablation
Use an augmentation that destroys class identity, such as very aggressive masking. Does downstream structure improve or collapse?

## Ablation table

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
1. What problem existed before this paper?
2. What was actually new?
3. What evidence did your notebook reproduce?
4. What does the scale gap prevent you from claiming?
5. Which idea survived into modern systems?
6. What would you test next?